In [1]:
import pandas as pd
import pyarrow 
import openpyxl
from unidecode import unidecode

## Tratamento dos dados

In [3]:
# Leitura dos dados em excel

df1 = pd.read_excel('Dados-iniciais/Referencia-pesa-municipio-v2.xlsx')
df2 = pd.read_excel('Dados-iniciais/Pesa-enderecos-ms-v2.xlsx', usecols=['PESA_MUNI','NOME_PESA','ENDERECO','N'])

In [5]:
# Normatização de texto

## Remover espaços quebras de linha
df2['PESA_MUNI'] = df2['PESA_MUNI'].str.replace('\n', ' ', regex=False)
df2['NOME_PESA'] = df2['NOME_PESA'].str.replace('\n', ' ', regex=False)
df2['ENDERECO'] = df2['ENDERECO'].str.replace('\n', ' ', regex=False)



In [ ]:
# Concatemnar endereço completo dos PESA df2
df2["ENDERECO_COMPLETO"] = (
    df2["ENDERECO"].astype(str).str.strip()
    + ", " +
    df2["N"].astype(str).str.strip()
    + ", " +
    df2["PESA_MUNI"].astype(str).str.strip()
    + ", SP, Brasil"
)

In [ ]:
# Geocodificar endereços do PESA df2

from geopy.geocoders import Nominatim
from time import sleep

geolocator = Nominatim(user_agent="pesa_geocoder_sp")

def geocode_address(endereco):
    try:
        location = geolocator.geocode(endereco, timeout=10)
        if location:
            return location.latitude, location.longitude
        return None, None
    except Exception:
        return None, None

df2["lat"] = None
df2["lon"] = None

for i, row in df2.iterrows():
    lat, lon = geocode_address(row["ENDERECO_COMPLETO"])
    df2.at[i, "lat"] = lat
    df2.at[i, "lon"] = lon
    sleep(1)




In [15]:
# Geocodificar municipios df1

# inicializar geocoder
geolocator = Nominatim(user_agent="geo_municipios_sp")

# função de limpeza (remove acento e padroniza)
def limpar_nome(muni):
    if pd.isna(muni):
        return None
    return unidecode(str(muni).strip())

# função de geocodificação
def geocode_municipio(muni):
    try:
        busca = f"{muni}, SP, Brasil"
        location = geolocator.geocode(busca, timeout=10)
        
        if location:
            return location.latitude, location.longitude
        return None, None
    
    except:
        return None, None

# aplicar limpeza
df1["muni_limpo"] = df1["REFERENCIADO"].apply(limpar_nome)

# lista única de municípios
municipios = df1["muni_limpo"].dropna().drop_duplicates()

# dicionário para armazenar coordenadas
coords = {}

print("Geocodificando municípios...")

for muni in municipios:
    lat, lon = geocode_municipio(muni)
    coords[muni] = (lat, lon)
    sleep(1)  # respeitar limite do Nominatim

print("Geocodificação concluída.")

# adicionar ao dataframe
df1["lat_muni"] = df1["muni_limpo"].map(lambda x: coords.get(x, (None, None))[0])
df1["lon_muni"] = df1["muni_limpo"].map(lambda x: coords.get(x, (None, None))[1])

Geocodificando municípios...
Geocodificação concluída.


In [ ]:
# Salva o arquivo com coordenadas
df1.to_excel('Dados-iniciais/Referencia-pesa-municipio-v3.xlsx')